# Explore and Discover

This notebook is your playground. You have already built a complete galaxy modelling pipeline — now we will use it to explore real physics.

Each experiment below asks you to change something and observe what happens. There are guiding questions, but **do not just answer them and move on** — try to understand *why* you see what you see.

---

## Setup

Run this cell first — it imports everything and re-uses the galaxy you built in Notebook 4.

In [ ]:
import sys, os
sys.path.insert(0, ".")
from helpers import (
    download_tng_galaxy, load_tng_particles,
    build_synthesizer_galaxy, get_spectrum,
    make_image, plot_spectrum, plot_one_image,
    plot_rgb, plot_particles,
)

import numpy as np
import matplotlib.pyplot as plt
from synthesizer.instruments import FilterCollection

# ============================================================
# PASTE YOUR API KEY HERE AGAIN
# ============================================================
API_KEY = "YOUR_API_KEY_HERE"
# ============================================================

print("Ready! Starting with Galaxy 553837 — run Notebook 4 first if you haven't already.")

In [ ]:
# Load and build the default galaxy (same as Notebook 4)
# If you already ran Notebook 4 this will be fast (data is cached)

GALAXY_ID = 553837

data_file    = download_tng_galaxy(GALAXY_ID, api_key=API_KEY)
particles    = load_tng_particles(data_file)
galaxy, grid = build_synthesizer_galaxy(particles)

print("\nRunning Synthesizer...")
sed, spectrum_label = get_spectrum(galaxy, grid)
print("Done!")

---

## Experiment 1 — Different Viewing Angles

The TNG simulation stores 3D positions for every particle. We can project the galaxy along any axis — x, y, or z — to simulate seeing it from different directions.

A **face-on** galaxy (disk pointing toward us) looks like a circle.
An **edge-on** galaxy (disk at 90°) looks like a thin line with a bright centre.

In [ ]:
# The helpers.py make_image() function projects along the z-axis by default.
# To change viewing angle we need to swap the coordinates.

from helpers import build_synthesizer_galaxy
from synthesizer.instruments import FilterCollection, Instrument
from unyt import kpc

def make_image_projected(particles, grid, axis='z',
                         fov_kpc=60, pixel_kpc=0.6,
                         filter_code="SLOAN/SLOAN.r"):
    """
    Make an image projecting along a given axis.
    axis = 'x', 'y', or 'z'
    """
    # Make a copy of particles with axes rotated
    p2 = {k: v.copy() for k, v in particles.items()}
    if axis == 'x':
        # look along x → plot y vs z
        p2['coords_kpc'] = particles['coords_kpc'][:, [1, 2, 0]]
    elif axis == 'y':
        # look along y → plot x vs z
        p2['coords_kpc'] = particles['coords_kpc'][:, [0, 2, 1]]
    # z (default): use coordinates as-is

    gal, _ = build_synthesizer_galaxy(p2, grid_dir='./grids',
                                       grid_name='maraston24-Te00_kroupa-0.1,100')
    _, spec_label = get_spectrum(gal, grid)
    imgs = make_image(gal, spec_label, fov_kpc=fov_kpc, pixel_kpc=pixel_kpc,
                      filter_codes=[filter_code])
    return imgs[filter_code].arr

print("Function ready!")

In [ ]:
# Make images from three perpendicular viewing directions
# (This will take a couple of minutes — Synthesizer is running three times!)

print("Making face-on image (project along z)...")
img_z = make_image_projected(particles, grid, axis='z')

print("Making side-on image (project along x)...")
img_x = make_image_projected(particles, grid, axis='x')

print("Making other side-on image (project along y)...")
img_y = make_image_projected(particles, grid, axis='y')

print("\nAll done!")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='black')

for ax, img, label in zip(axes,
    [img_z, img_x, img_y],
    ["Face-on (z-axis)", "Edge-on (x-axis)", "Edge-on (y-axis)"]):
    arr = np.where(img > 0, np.log10(img + 1e-40), np.nan)
    vmin = np.nanpercentile(arr, 5)
    vmax = np.nanpercentile(arr, 99.5)
    ax.imshow(arr, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
    ax.set_title(label, color='white', fontsize=11)
    ax.axis('off')
    ax.set_facecolor('black')

fig.patch.set_facecolor('black')
fig.suptitle(f"Galaxy {GALAXY_ID} — Same galaxy, three viewing angles",
             color='white', fontsize=13)
plt.tight_layout()
plt.show()

### Questions for Experiment 1

1. Is galaxy 553837 a flat disk (like a frisbee) or a sphere?
   *Hint: look at the shapes in the three views. A sphere looks the same from all angles; a disk looks circular face-on and like a line edge-on.*

2. In real astronomy we cannot choose our viewing angle — we can only see each real galaxy from one direction. Does this make it harder to measure galaxy properties (like size)? Why?

3. If you were looking at a real photograph of a galaxy and could not tell if it was face-on or edge-on, how might that cause problems?

---

## Experiment 2 — What Different Wavelengths Reveal

Now let's compare the same galaxy across a wide range of wavelengths, from the far ultraviolet to the near-infrared.

Remember:
- **UV light** comes from **hot, young, massive stars** — galaxies that are actively forming stars
- **Red/infrared light** comes from **cool, old, low-mass stars** — galaxies with an established older population
- **The distribution** of light tells you about the galaxy's **star formation history**

In [ ]:
# Make images in a range of filters
# (We use the default z-projection — face-on)

from synthesizer.instruments import FilterCollection, Instrument
from unyt import kpc as kpc_unit

bands = [
    ("GALEX/GALEX.FUV",   "purple",  "Far UV (~1540 Å)"),
    ("GALEX/GALEX.NUV",   "violet",  "Near UV (~2300 Å)"),
    ("SLOAN/SLOAN.u",     "blue",    "u-band (~3550 Å)"),
    ("SLOAN/SLOAN.g",     "green",   "g-band (~4770 Å)"),
    ("SLOAN/SLOAN.r",     "orange",  "r-band (~6230 Å)"),
    ("SLOAN/SLOAN.i",     "red",     "i-band (~7630 Å)"),
    ("2MASS/2MASS.J",     "darkred", "J-band (~1.2 μm)"),
    ("2MASS/2MASS.K",     "maroon",  "K-band (~2.2 μm)"),
]

all_filter_codes = [b[0] for b in bands]

print("Making images in 8 filters (this will take a few minutes)...")
all_images = make_image(
    galaxy, spectrum_label,
    fov_kpc=60, pixel_kpc=0.6,
    filter_codes=all_filter_codes
)
print("Done!")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9), facecolor='black')
axes = axes.flatten()

for ax, (code, colour, label) in zip(axes, bands):
    arr = np.array(all_images[code].arr)
    arr_log = np.where(arr > 0, np.log10(arr + 1e-40), np.nan)
    vmin = np.nanpercentile(arr_log, 5)
    vmax = np.nanpercentile(arr_log, 99.5)
    ax.imshow(arr_log, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
    ax.set_title(label, color=colour, fontsize=9.5, pad=4)
    ax.axis('off')
    ax.set_facecolor('black')

fig.patch.set_facecolor('black')
fig.suptitle(f"Galaxy {GALAXY_ID} — UV to Infrared",
             color='white', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Questions for Experiment 2

1. Compare the **UV image** with the **K-band (infrared) image**. Are they the same shape? If not, why not?
   *Hint: think about where young stars form in a spiral galaxy (outer arms) vs where old stars live (the bulge in the centre).*

2. If you could only observe this galaxy through one filter, which would you choose to best measure its **total stellar mass** (all the stars it has ever made)?
   *Hint: infrared light is less affected by young massive stars and dust.*

3. If you could only observe through one filter to measure the **current star formation rate** (how many new stars are forming right now), which would you choose?

---

## Experiment 3 — What If There Were No Dust?

Dust absorbs starlight — especially UV. We can test the effect of dust by looking at the spectrum with and without it.

In this experiment we use the **Calzetti attenuation law** — a standard model for how dust affects galaxy spectra. The key parameter is **A_V**: the amount of attenuation at the V-band (5500 Å).

- A_V = 0 → no dust
- A_V = 1 → about 60% of V-band light is absorbed
- A_V = 3 → about 94% of V-band light is absorbed

In [ ]:
# Apply different amounts of dust attenuation to the spectrum

from synthesizer.emission_models.attenuation import CalzettiAtt
from synthesizer.emission_models import AttenuatedEmission, StellarEmissionModel

def get_attenuated_spectrum(galaxy, grid, A_V):
    """
    Return the galaxy spectrum attenuated by dust with V-band attenuation A_V.
    A_V = 0 means no dust.
    """
    from unyt import dimensionless
    stellar_model = StellarEmissionModel(grid)
    dust_model    = CalzettiAtt(A_V=A_V)
    att_model     = AttenuatedEmission(emitter=stellar_model, attenuation=dust_model)
    galaxy.get_spectra(att_model)
    available = list(galaxy.stars.spectra.keys())
    # Attenuated spectrum is typically labelled 'attenuated'
    for key in available:
        if 'attenuated' in key.lower() or 'dust' in key.lower():
            return galaxy.stars.spectra[key]
    # Fallback: return the last generated one
    return galaxy.stars.spectra[available[-1]]

# Compare no dust vs heavy dust
wavelengths_A = sed.lam.to("Angstrom").value

A_V_values = [0.0, 0.5, 1.0, 2.0]
colours = ['royalblue', 'green', 'orange', 'red']

fig, ax = plt.subplots(figsize=(12, 5))

# Intrinsic (no dust)
ax.loglog(wavelengths_A, sed.luminosity, color='royalblue',
          linewidth=2, label='A_V = 0 (no dust — intrinsic)', alpha=0.9)

# Apply dust manually using the Calzetti law
def apply_calzetti(wavelengths_A, luminosity, A_V):
    w_um = wavelengths_A / 10000
    k = np.where(
        wavelengths_A < 6300,
        2.659 * (-2.156 + 1.509/w_um - 0.198/w_um**2 + 0.011/w_um**3) + 4.05,
        2.659 * (-1.857 + 1.040/w_um) + 4.05
    )
    k = np.clip(k, 0, None)
    return luminosity * 10 ** (-0.4 * A_V * k / 4.05)

lum = sed.luminosity
for A_V, colour in zip([0.5, 1.0, 2.0], ['green', 'orange', 'red']):
    lum_att = apply_calzetti(wavelengths_A, lum, A_V)
    ax.loglog(wavelengths_A, lum_att, color=colour, linewidth=2,
              label=f'A_V = {A_V} mag')

ax.axvspan(3800, 7000, alpha=0.08, color='gold', label='Visible light')
ax.set_xlabel('Wavelength (Å)', fontsize=12)
ax.set_ylabel(r'Luminosity (erg s$^{-1}$ Hz$^{-1}$)', fontsize=12)
ax.set_title(f'Galaxy {GALAXY_ID} — Effect of Dust', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print("Notice: dust has a much bigger effect at UV (left) than at infrared (right).")
print("This is because dust grains are physically small — about the same size as")
print("UV photons' wavelength — so they scatter/absorb UV much more efficiently.")

### Questions for Experiment 3

1. If a real astronomer measures a galaxy's UV luminosity and uses it to estimate the star formation rate, but does not correct for dust — will they **over-estimate** or **under-estimate** the true rate?

2. The Milky Way has a V-band attenuation of roughly A_V = 0.5 mag toward most of its disk. Compared to the spectrum with no dust, approximately how much UV light is lost?

3. Astronomers often use a quantity called the **UV slope** (how steep the UV spectrum is) as a way to estimate dust. Looking at your plot, does adding more dust make the UV slope steeper or shallower? Why?

---

## Experiment 4 — Compare Different Galaxies

Now let's compare a few different galaxies. They all live in the same TNG50 simulation, but they have different histories — different star formation rates, different amounts of merging, different environments.


In [ ]:
# Download and build a second galaxy for comparison
# This one is a more massive, older 'red and dead' galaxy

GALAXY_ID_2 = 385350   # try also: 468590, 474170, 406595

print(f"Downloading galaxy {GALAXY_ID_2}...")
data_file_2    = download_tng_galaxy(GALAXY_ID_2, api_key=API_KEY)
particles_2    = load_tng_particles(data_file_2)
galaxy_2, grid2 = build_synthesizer_galaxy(particles_2)
print("\nRunning Synthesizer on galaxy 2...")
sed_2, label_2  = get_spectrum(galaxy_2, grid2)
print("Done!")

In [ ]:
# Compare the spectra of the two galaxies

lam_A = sed.lam.to("Angstrom").value

fig, ax = plt.subplots(figsize=(12, 5))

lum1_norm = sed.luminosity / sed.luminosity.max()
lum2_norm = sed_2.luminosity / sed_2.luminosity.max()

ax.loglog(lam_A, lum1_norm, color='steelblue', lw=2,
          label=f'Galaxy {GALAXY_ID}')
ax.loglog(lam_A, lum2_norm, color='tomato', lw=2,
          label=f'Galaxy {GALAXY_ID_2}')

ax.axvspan(3800, 7000, alpha=0.1, color='gold', label='Visible')
ax.set_xlabel('Wavelength (Å)', fontsize=12)
ax.set_ylabel('Normalised luminosity', fontsize=12)
ax.set_title('Spectral comparison: two TNG50 galaxies', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

In [ ]:
# Compare particle properties side by side

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, p, gid, colour in zip(axes,
    [particles, particles_2],
    [GALAXY_ID, GALAXY_ID_2],
    ['steelblue', 'tomato']):
    ax.hist(p['ages_yr'] / 1e9, bins=40,
            weights=p['masses_msun'], density=True,
            color=colour, alpha=0.7, edgecolor='white', linewidth=0.5)
    ax.axvline(np.average(p['ages_yr']/1e9, weights=p['masses_msun']),
               color='gold', linewidth=2.5, linestyle='--',
               label=f"Mean age = {np.average(p['ages_yr']/1e9, weights=p['masses_msun']):.1f} Gyr")
    ax.set_xlabel('Stellar age (Gyr)', fontsize=12)
    ax.set_ylabel('Mass fraction', fontsize=12)
    ax.set_title(f'Galaxy {gid} — age distribution', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey question: which galaxy is forming more stars right now?")
print("(A galaxy still forming stars has many young, recently-formed particles.)")

In [ ]:
# Make images of both and compare side-by-side

print("Making images of both galaxies...")
imgs1 = make_image(galaxy,   spectrum_label, fov_kpc=60, pixel_kpc=0.6,
                   filter_codes=["SLOAN/SLOAN.g", "SLOAN/SLOAN.r", "SLOAN/SLOAN.u"])
imgs2 = make_image(galaxy_2, label_2,        fov_kpc=60, pixel_kpc=0.6,
                   filter_codes=["SLOAN/SLOAN.g", "SLOAN/SLOAN.r", "SLOAN/SLOAN.u"])
print("Done!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6), facecolor='black')

for ax, imgs, gid, title in zip(
    axes,
    [imgs1, imgs2],
    [GALAXY_ID, GALAXY_ID_2],
    [f"Galaxy {GALAXY_ID}", f"Galaxy {GALAXY_ID_2}"]
):
    # Quick single-band display (r-band)
    arr = np.array(imgs["SLOAN/SLOAN.r"].arr)
    arr_log = np.where(arr > 0, np.log10(arr + 1e-40), np.nan)
    vmin = np.nanpercentile(arr_log, 5)
    vmax = np.nanpercentile(arr_log, 99.5)
    ax.imshow(arr_log, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
    ax.set_title(title + "\n(r-band)", color='white', fontsize=12)
    ax.axis('off')
    ax.set_facecolor('black')

fig.patch.set_facecolor('black')
fig.suptitle("Two TNG50 galaxies compared", color='white', fontsize=14)
plt.tight_layout()
plt.show()

### Questions for Experiment 4

1. Look at the age distributions. Which galaxy is younger on average? Does this match what you see in the spectra?

2. Look at the images. Are the two galaxies the same shape? What might cause differences in morphology (shape)?

3. How would an astronomer classify these two galaxies if they saw them in a real telescope image — as 'blue' or 'red'? What does that tell you about their history?

---

## Experiment 5 — Your Own Investigation

Now it's your turn to design an experiment. Here are some ideas:

**Idea A:** Try a completely different galaxy. Explore the list of subhalo IDs in TNG50 — you could find almost any type of galaxy.
Some interesting ones to try: `406595`, `474170`, `517946`, `590803`, `624150`

**Idea B:** Compute the colour of a galaxy — the difference in magnitude between two filters. 
For example, `g - r` colour is a classic measure of how 'red' or 'blue' a galaxy is.

**Idea C:** Look at a galaxy's stellar mass — how much of its total stellar mass is in young stars (< 1 Gyr) vs old stars (> 5 Gyr)?

**Idea D:** Change the image resolution (`pixel_kpc`) and field of view (`fov_kpc`) to zoom in on a galaxy's bright centre (the 'bulge').

Use the cells below as scratch space:

In [ ]:
# ===================================================
# YOUR INVESTIGATION — start here!
# ===================================================

# Change the subhalo ID below to explore a new galaxy
MY_GALAXY_ID = 406595   # ← try different IDs!

data_file_mine    = download_tng_galaxy(MY_GALAXY_ID, api_key=API_KEY)
particles_mine    = load_tng_particles(data_file_mine)
galaxy_mine, grid_mine = build_synthesizer_galaxy(particles_mine)
sed_mine, label_mine   = get_spectrum(galaxy_mine, grid_mine)

print("\nYour galaxy properties:")
print(f"  Stellar mass: {particles_mine['masses_msun'].sum():.2e} Msun")
print(f"  Mean age:     {np.average(particles_mine['ages_yr']/1e9, weights=particles_mine['masses_msun']):.2f} Gyr")
print(f"  Mean metallicity: {np.average(particles_mine['metallicity'], weights=particles_mine['masses_msun']):.5f}")

# Compare to the Milky Way: ~6 × 10^10 Msun, ~8 Gyr mean stellar age
print()
print("For comparison, the Milky Way: ~6e10 Msun, ~8 Gyr mean age")

In [ ]:
# Make an image of your galaxy — try changing fov_kpc and pixel_kpc!

my_images = make_image(
    galaxy_mine, label_mine,
    fov_kpc    = 60,    # ← try 30, 80, 120 kpc
    pixel_kpc  = 0.6,  # ← try 0.3 (sharper) or 1.0 (coarser)
    filter_codes = ["SLOAN/SLOAN.r", "SLOAN/SLOAN.g", "SLOAN/SLOAN.u"],
)

plot_rgb(
    my_images,
    r_filter="SLOAN/SLOAN.r",
    g_filter="SLOAN/SLOAN.g",
    b_filter="SLOAN/SLOAN.u",
    title=f"My galaxy ({MY_GALAXY_ID}) — RGB",
    stretch=0.008
)

In [ ]:
# Compute the g-r colour (a classic galaxy colour indicator)
# Positive = red (old stars); Negative = blue (young stars)

fc = FilterCollection(filter_codes=["SLOAN/SLOAN.g", "SLOAN/SLOAN.r"])
fc.calc_pivot_lam()
sed_mine.get_photo_luminosities(fc)
phot = sed_mine.photo_luminosities

lum_g = float(phot["SLOAN/SLOAN.g"])
lum_r = float(phot["SLOAN/SLOAN.r"])

# Convert to magnitudes: m = -2.5 * log10(flux) + constant
# For the ratio, the constant cancels:
g_minus_r = -2.5 * np.log10(lum_g / lum_r)

print(f"Galaxy {MY_GALAXY_ID} g-r colour = {g_minus_r:.3f}")
print()
if g_minus_r > 0.6:
    print("This is a RED galaxy — dominated by old stars, probably not forming many new stars.")
elif g_minus_r > 0.3:
    print("This is a GREEN galaxy — a mix of young and old stars; possibly transitioning.")
else:
    print("This is a BLUE galaxy — lots of young stars, actively forming new ones right now.")

# Compare the two galaxies from Experiment 4
fc_ref = FilterCollection(filter_codes=["SLOAN/SLOAN.g", "SLOAN/SLOAN.r"])
fc_ref.calc_pivot_lam()
sed.get_photo_luminosities(fc_ref)
sed_2.get_photo_luminosities(fc_ref)

for g, s, name in [(GALAXY_ID, sed, 'Galaxy 1'), (GALAXY_ID_2, sed_2, 'Galaxy 2')]:
    p = s.photo_luminosities
    gmr = -2.5 * np.log10(float(p['SLOAN/SLOAN.g']) / float(p['SLOAN/SLOAN.r']))
    print(f"{name} ({g}): g-r = {gmr:.3f}")

In [ ]:
# Free exploration cell — use this however you like!
# Some suggestions:
# - Plot the spatial distribution of young vs old stars
# - Try a very different pixel_kpc to see how resolution changes the image
# - Compute the fraction of mass in stars younger than 1 Gyr

# Young star fraction
young_mask = particles_mine['ages_yr'] < 1e9   # younger than 1 Gyr
young_frac = particles_mine['masses_msun'][young_mask].sum() / particles_mine['masses_msun'].sum()

print(f"Fraction of stellar mass in stars younger than 1 Gyr: {young_frac*100:.1f}%")
print()
print("For comparison:")
print("  A starburst galaxy: ~20–50%")
print("  A normal spiral like the Milky Way: ~5–10%")
print("  A quenched elliptical galaxy: < 1%")

---

## Final Challenge — Build a Colour-Magnitude Diagram

A **colour-magnitude diagram** (CMD) shows galaxies plotted by how bright they are vs. how red or blue they are. It is one of the most fundamental plots in galaxy astronomy — the real version of this, made with thousands of real galaxies, is called the **Red Sequence** (old red galaxies) vs the **Blue Cloud** (young blue galaxies).

Let's make a mini version using three galaxies.

In [ ]:
# We already have SEDs for three galaxies:
# galaxy (553837), galaxy_2 (385350), galaxy_mine (yours)
# Let's compute magnitude and colour for each

results = []

for gal_id, s, mstar in [
    (GALAXY_ID,    sed,      particles['masses_msun'].sum()),
    (GALAXY_ID_2,  sed_2,    particles_2['masses_msun'].sum()),
    (MY_GALAXY_ID, sed_mine, particles_mine['masses_msun'].sum()),
]:
    fc_cmd = FilterCollection(filter_codes=["SLOAN/SLOAN.g", "SLOAN/SLOAN.r"])
    fc_cmd.calc_pivot_lam()
    s.get_photo_luminosities(fc_cmd)
    p = s.photo_luminosities
    gmr = -2.5 * np.log10(float(p['SLOAN/SLOAN.g']) / float(p['SLOAN/SLOAN.r']))
    lum_r = np.log10(float(p['SLOAN/SLOAN.r']))
    results.append((gal_id, gmr, lum_r, mstar))

# Plot the colour-magnitude diagram
fig, ax = plt.subplots(figsize=(9, 7))

# Background: illustrative distribution of real galaxies
np.random.seed(42)
n_bg = 500
bg_lum  = np.random.uniform(27, 32, n_bg)
# Red sequence
bg_col_red  = 0.75 + np.random.normal(0, 0.05, n_bg//2)
bg_lum_red  = np.random.normal(31, 0.7, n_bg//2)
# Blue cloud
bg_col_blue = 0.2 + np.random.normal(0, 0.1, n_bg//2)
bg_lum_blue = np.random.normal(29.5, 0.8, n_bg//2)
ax.scatter(bg_col_red,  bg_lum_red,  s=6, color='lightcoral',  alpha=0.4, label='Typical red galaxies')
ax.scatter(bg_col_blue, bg_lum_blue, s=6, color='lightsteelblue', alpha=0.4, label='Typical blue galaxies')

# Our galaxies
colours_pts = ['steelblue', 'tomato', 'gold']
for (gid, gmr, lum_r, mstar), colour in zip(results, colours_pts):
    ax.scatter(gmr, lum_r, s=250, color=colour, zorder=5, edgecolors='white', linewidths=1.5)
    ax.annotate(f"Galaxy\n{gid}", (gmr, lum_r), textcoords='offset points',
                xytext=(10, 5), color=colour, fontsize=10, fontweight='bold')

ax.set_xlabel('g − r colour  (redder →)', fontsize=13)
ax.set_ylabel(r'log$_{10}$ r-band luminosity  (brighter ↑)', fontsize=13)
ax.set_title('Galaxy Colour-Magnitude Diagram', fontsize=14)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='Blue/Red boundary')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Where do your galaxies sit? Are they in the blue cloud or on the red sequence?")
print("Can you explain why, based on their stellar age distributions?")

---

## What You Have Accomplished Today

In one day you have gone from knowing no Python at all to:

- Writing Python code with variables, loops, functions, and plots
- Understanding why astronomers need computers
- Learning what cosmological simulations (IllustrisTNG) are and how they work
- Understanding the concept of **forward modelling**
- Running **Synthesizer** — professional astrophysics software — on real simulation data
- Producing genuine multi-wavelength galaxy images
- Exploring how galaxy properties (age, metallicity, dust, viewing angle) affect their appearance
- Constructing a colour-magnitude diagram and placing galaxies on it

This is genuinely what research astronomers do — you have just done a miniature version of the work in Sophie's PhD.

---

## Where Next?

If you want to learn more:

- **Python**: *Python Crash Course* by Eric Matthes; free course at https://www.learnpython.org
- **Astronomy**: *The Feynman Lectures on Physics* (free online) for the fundamentals; ESA and NASA websites for current missions
- **Synthesizer**: https://docs.synthesizer.space — the full documentation
- **IllustrisTNG**: https://www.tng-project.org — explore all the simulations; you can query galaxy properties directly
- **University**: Durham, Cambridge, Oxford, Edinburgh, UCL all have strong astrophysics departments — A-levels in Maths and Physics are the key requirement

---

*Thank you for visiting the ICC. We hope you enjoyed your day!*